In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/india_professional_skills_intelligence.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (12000, 26)


,profile_id,current_job_title,seniority_level,industry,primary_skill_domain,top_skills,skill_count,years_experience,education_level,degree_field,city,state,company_size,employment_type,work_mode,is_gig_worker,uses_generative_ai_tools,certifications_count,endorsement_count,connections_tier,profile_completeness_pct,open_to_work,last_profile_update,estimated_annual_salary_lpa,employability_score,in_demand_skill_flag
0,IN-PROF-208385,Junior Product Manager,Entry-level,Manufacturing,Product Management,Product Strategy;User Research;Roadmapping;Sta...,7,0.7,Master's,Business Administration,New Delhi,Delhi,Large (501-5000),Contract,Hybrid,No,Yes,2,25.0,150-500,65.0,No,1-3 months ago,5.8,43.6,No
1,IN-PROF-200677,Lead Software Engineer,Lead/Principal,Technology,Software Engineering,Python;Git;Microservices;JavaScript,4,14.1,Bachelor's,Mechanical,Hyderabad,Telangana,Startup (1-50),Full-time,On-site,No,Yes,1,13.0,150-500,100.0,No,6+ months ago,50.0,59.9,No
2,IN-PROF-203340,UX Designer,Mid-Senior,Manufacturing,UI/UX Design,Design Systems;Wireframing;Prototyping;Figma,4,5.5,Master's,Life Sciences,Pune,Maharashtra,Enterprise/MNC (5000+),Full-time,Hybrid,No,Yes,1,94.0,500+,90.0,No,6+ months ago,17.0,48.0,No
3,IN-PROF-204452,Lead Marketing Specialist,Lead/Principal,Technology,Digital Marketing,Content Marketing;SEM;Performance Marketing;SE...,7,12.0,Diploma,Business Administration,Hyderabad,Telangana,Enterprise/MNC (5000+),Full-time,On-site,No,Yes,3,245.0,150-500,87.0,Yes,This month,48.0,56.9,No
4,IN-PROF-211658,Product Manager,Mid-Senior,Telecom,Product Management,Scrum;Agile;JIRA;Product Strategy;User Researc...,6,7.1,Bachelor's,Computer Science/IT,Noida,Uttar Pradesh,Startup (1-50),Contract,On-site,Yes,Yes,0,120.0,1000+,61.0,No,3-6 months ago,16.2,59.5,No


In [3]:
# Dataset information
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Number of rows: 12000
Number of columns: 26

Column Names:
['profile_id', 'current_job_title', 'seniority_level', 'industry', 'primary_skill_domain', 'top_skills', 'skill_count', 'years_experience', 'education_level', 'degree_field', 'city', 'state', 'company_size', 'employment_type', 'work_mode', 'is_gig_worker', 'uses_generative_ai_tools', 'certifications_count', 'endorsement_count', 'connections_tier', 'profile_completeness_pct', 'open_to_work', 'last_profile_update', 'estimated_annual_salary_lpa', 'employability_score', 'in_demand_skill_flag']

Data Types:
profile_id                         str
current_job_title                  str
seniority_level                    str
industry                           str
primary_skill_domain               str
top_skills                         str
skill_count                      int64
years_experience               float64
education_level                    str
degree_field                       str
city                               str
stat

In [4]:
# Inspect important categorical columns

categorical_columns = [
    "seniority_level",
    "industry",
    "education_level",
    "company_size",
    "employment_type",
    "work_mode",
    "is_gig_worker",
    "uses_generative_ai_tools",
    "connections_tier",
    "open_to_work",
    "in_demand_skill_flag"
]

for col in categorical_columns:
    print(f"\n{'='*50}")
    print(f"{col}")
    print(df[col].value_counts(dropna=False))


seniority_level
seniority_level
Entry-level       3185
Associate         2885
Mid-Senior        2675
Senior            1786
Lead/Principal    1010
Director+          459
Name: count, dtype: int64

industry
industry
Technology                  2645
BFSI                        1751
Manufacturing               1410
Retail/E-commerce           1097
Healthcare                  1058
Renewable Energy             729
Consulting                   708
Education                    585
Telecom                      548
Government/Public Sector     482
Media & Entertainment        445
Logistics & Supply Chain     438
TECHNOLOGY                    22
RETAIL/E-COMMERCE             12
MANUFACTURING                 12
HEALTHCARE                    11
EDUCATION                     10
RENEWABLE ENERGY              10
MEDIA & ENTERTAINMENT          7
CONSULTING                     7
TELECOM                        5
LOGISTICS & SUPPLY CHAIN       5
GOVERNMENT/PUBLIC SECTOR       3
Name: count, dtype: int64

In [5]:
# Summary statistics for numerical columns

df.describe().T

,count,mean,std,min,25%,50%,75%,max
skill_count,12000.0,5.614333,1.209704,4.0,5.0,5.0,7.0,8.0
years_experience,12000.0,5.992467,5.625784,0.0,1.9,4.0,8.6,30.0
certifications_count,12000.0,1.542000,1.279338,0.0,1.0,1.0,2.0,9.0
endorsement_count,11760.0,165.159269,214.926351,0.0,40.0,95.0,207.0,3081.0
profile_completeness_pct,12000.0,66.259833,13.563775,15.0,57.0,66.0,76.0,100.0
estimated_annual_salary_lpa,12000.0,20.694258,19.494583,2.8,7.6,13.5,26.7,166.1
employability_score,12000.0,49.190842,13.638939,2.1,39.9,49.2,58.3,99.0


In [6]:
# Normalize industry: keep the most frequent spelling of each category
counts = df["industry"].str.strip().value_counts()
lookup = {}
for v in counts.index:               # most frequent first
    lookup.setdefault(v.lower(), v)
df["industry"] = df["industry"].str.strip().str.lower().map(lookup)
print(df["industry"].nunique(), "industries")   # expect 12

# Missing values
df["endorsement_count"] = df["endorsement_count"].fillna(df["endorsement_count"].median())

12 industries


In [7]:
print(pd.crosstab(df["education_level"], df["degree_field"].isna()))

degree_field                 False  True 
education_level                          
Bachelor's                    6810    225
Bootcamp/Certification-only    708     15
Diploma                        917     22
Master's                      2979     89
PhD                            226      9


In [8]:
num = df.select_dtypes("number")
print(num.corr()[["employability_score", "estimated_annual_salary_lpa"]].round(2))

print(df.groupby("in_demand_skill_flag")["employability_score"].mean())
print(pd.crosstab(df["employment_type"], df["is_gig_worker"]))
print(df.groupby("seniority_level")["years_experience"].describe().round(1))

                             employability_score  estimated_annual_salary_lpa
skill_count                                 0.04                         0.01
years_experience                            0.37                         0.92
certifications_count                        0.28                         0.15
endorsement_count                           0.04                         0.10
profile_completeness_pct                    0.21                         0.38
estimated_annual_salary_lpa                 0.37                         1.00
employability_score                         1.00                         0.37
in_demand_skill_flag
No     47.787558
Yes    54.544944
Name: employability_score, dtype: float64
is_gig_worker      No  Yes
employment_type           
Contract          803  253
Freelance/Gig       0  918
Full-time        9659    0
Internship        367    0
                  count  mean  std   min   25%   50%   75%   max
seniority_level                                     